In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# NB_Config_PROC_B1: the one config of the colocated Procurement Bronze (workspace SAP_DAB_BRZ_PROD)
#   Read by NB_Bronze_PROC_B1 (Bronze2 load), NB_PROC_B1_Swap (B1 swap), NB_PROC_B1_Copy (optional backup)
#   and NB_Check_B1_Keys. Lean on purpose: Bronze settings and keys only. Columns, renames, joins and enrichment
#   belong to Silver, whose configs (NB_Config_PROC) are unchanged.
#   Generated 23 Nov 2026 from NB_Config_PROC (11 tables); edit by hand from here on.
# ════════════════════════════════════════════════════════════════════════════

# ── 1. Run settings ─────────────────────────────────────────────────────────────
# Variable Library VL_PROC_Bronze2: B1 = Dab's lakehouses (read and swapped), B2 = ours (Bronze2 + every log).
# Every value is a lakehouse ABFS path ending in /Tables; tables are under dbo/.
PIPELINE   = 'PROC_B1'
VL_NAME    = 'VL_PROC_Bronze2'
B2_KEY     = 'B2_PROC_LH_ABFSS'              # our Bronze2 lakehouse
LAKEHOUSES = {
    'bronze': B2_KEY,                 # Bronze2 tables
    'ops':    'bronze',               # run, table and DQ logs: same lakehouse (no separate OPS lakehouse)
}

# Bronze source: B1. A table's source_lh_key / source_schema override these.
SOURCE_LH_KEY    = 'B1_PROC_LH_ABFSS'
SOURCE_SCHEMA    = 'dbo'
BRONZE_MODE      = 'cdc'              # 'cdc' keeps history and soft deletes; 'overwrite' replaces the table
DESC_BRONZE_MODE = 'overwrite'        # description (text) tables

# ── 2. Swap (NB_PROC_B1_Swap): Dab writes <table>_stg next to the live B1 table, the swap switches it in ──
# Safeguards (NB_B1_Swap): dry run by default; staging must be complete (min_rows_ratio), left alone for quiet_minutes and
# newer than live; no lost or retyped columns; rename, verify, roll back from <table>_prev on any failed check.
DEFAULTS = {
    'tables_folder':    'dbo',        # between the VL value (…/Tables) and the tables
    'stg_suffix':       '_stg',       # staging folder = <table>_stg, the previous copy = <table>_prev
    'prev_suffix':      '_prev',
    'min_rows_ratio':   1.0,          # staging needs at least this share of the live rows (0.98 = up to 2% smaller)
    'quiet_minutes':    10,           # staging must have been left alone this long (a load may still be running)
    'delete_prev':      True,         # delete the previous copy once the new table is verified
    'require_new_data': True,         # False: also swap staging not written after the live table (testing only)
    'dry_run':          True,         # True: decide and log only. Set False to start swapping
    'parallel_tables':  1,            # tables swapped at the same time (domain level only)
    'range':            None,         # range swap (SWAP_SCOPE = 'range'): {'year_col', 'period_col', 'periods'}
}
MAX_PARALLEL_DOMAINS = 1
# b1_swap_log goes to B2 (our lakehouse), never into Dab's B1 lakehouse
LOG_TABLE_PATH = notebookutils.variableLibrary.getLibrary(VL_NAME)[B2_KEY].strip().rstrip('/') + '/dbo/b1_swap_log'

SWAP_DOMAINS = {
    'PROC': {
        'vl':     VL_NAME,
        'lh_key': SOURCE_LH_KEY,      # B1: the swap renames folders in Dab's lakehouse
        'parallel_tables': 4,
        'tables': [                        # CONFIRM against the team's NB_Config_PROC_B1_Swap list
        'EKKO', 'EKPO', 'EKET', 'EKBE', 'EKES', 'EKKN', 'EKPA', 'EKAB', 'EKPV', 'EBKN', 'EBAN'
        ],
    },
}

# ── 3. Backup copy (NB_PROC_B1_Copy, optional): not needed for the swap, which keeps its own rollback (<table>_prev) ──
# Empty = nothing to copy. To back up B1 first, add the backup lakehouse to the Variable Library (e.g. B1_PROC_BACKUP_LH_ABFSS)
# and an entry like:
#   {'name': 'PROC B1 backup', 'source_vl': VL_NAME, 'source_key': SOURCE_LH_KEY, 'source_folder': 'dbo',
#    'target_vl': VL_NAME, 'target_key': 'B1_PROC_BACKUP_LH_ABFSS', 'target_folder': 'dbo', 'tables': '*'},
COPIES = []
COPY_WORKERS = 4
COPY_DRY_RUN = True                   # True: list what would be copied and change nothing
COPY_ONLY    = ''                     # blank = every entry; or the start of entry names

# ── 4. Tables ───────────────────────────────────────────────────────────────────
# One line per table: primary_keys (SAP key, without MANDT), and only what differs from the settings above.
PROC_TABLE_CONFIGS = {                # 11 tables
    'EKKO': {'primary_keys': ['EBELN']},
    'EKPO': {'primary_keys': ['EBELN', 'EBELP']},
    'EKET': {'primary_keys': ['EBELN', 'EBELP', 'ETENR']},
    'EKBE': {'primary_keys': ['EBELN', 'EBELP', 'ZEKKN', 'VGABE', 'GJAHR', 'BELNR', 'BUZEI']},
    'EKES': {'primary_keys': ['EBELN', 'EBELP', 'ETENS']},
    'EKKN': {'primary_keys': ['EBELN', 'EBELP', 'ZEKKN']},
    'EKPA': {'primary_keys': ['EBELN', 'EKORG', 'LTSNR', 'WERKS', 'PARVW', 'PARZA']},
    'EKAB': {'primary_keys': ['KONNR', 'KTPNR', 'EBELN', 'EBELP']},
    'EKPV': {'primary_keys': ['EBELN', 'EBELP']},
    'EBKN': {'primary_keys': ['BANFN', 'BNFPO', 'ZEBKN']},
    'EBAN': {'primary_keys': ['BANFN', 'BNFPO']},
}